In [1]:
import os
from datetime import datetime, timezone
from pprint import pprint
from uuid import uuid4

from graphiti_core.driver.oracle_driver import OracleDriver
from graphiti_core.edges import EntityEdge
from graphiti_core.nodes import EntityNode

In [ ]:
async def create_oracle_graph() -> None:
    """Create a small graph through the Oracle driver and inspect the RDF table."""
    driver = OracleDriver(log_queries=True)

    if not driver.rdf_enabled:
        raise RuntimeError(
            'This notebook graph example uses Oracle RDF mode. '
            'Set ORACLE_USE_RDF=true and rerun the cell.'
        )

    created_at = datetime.now(timezone.utc)
    group_id = f'notebook-demo-{uuid4().hex[:8]}'

    alice = EntityNode(
        uuid=f'entity-{uuid4().hex[:8]}',
        name='Alice',
        group_id=group_id,
        labels=['Person'],
        created_at=created_at,
        summary='Demo person node created from the notebook.',
        attributes={'role': 'engineer', 'team': 'graph'},
    )
    bob = EntityNode(
        uuid=f'entity-{uuid4().hex[:8]}',
        name='Bob',
        group_id=group_id,
        labels=['Person'],
        created_at=created_at,
        summary='Second demo person node created from the notebook.',
        attributes={'role': 'manager', 'team': 'graph'},
    )
    works_with = EntityEdge(
        uuid=f'edge-{uuid4().hex[:8]}',
        group_id=group_id,
        source_node_uuid=alice.uuid,
        target_node_uuid=bob.uuid,
        created_at=created_at,
        name='works_with',
        fact='Alice works with Bob on the Graphiti Oracle integration.',
        episodes=[],
        valid_at=created_at,
        attributes={'confidence': 0.99, 'source': 'test.ipynb'},
    )

    try:
        connection_rows, _, _ = await driver.execute_query('SELECT 1 AS value FROM dual')
        print('Connection check:')
        pprint(connection_rows)

        rdf_table_name = driver.rdf_table_name()
        rdf_owner, rdf_table = rdf_table_name.split('.', maxsplit=1)

        before_rows, _, _ = await driver.execute_query(
            f'SELECT COUNT(*) AS triple_count FROM {rdf_table_name}'
        )
        print(f'\nTriple count before insert in {rdf_table_name}:')
        pprint(before_rows)

        await driver.entity_node_ops.save(driver, alice)
        await driver.entity_node_ops.save(driver, bob)
        await driver.entity_edge_ops.save(driver, works_with)

        print(f'\nCreated demo graph group: {group_id}')
        pprint(
            {
                'node_uuids': [alice.uuid, bob.uuid],
                'edge_uuid': works_with.uuid,
                'relation': works_with.fact,
            }
        )

        rdf_table_rows, _, _ = await driver.execute_query(
            """
            SELECT owner, table_name, tablespace_name
            FROM all_tables
            WHERE owner = $owner AND table_name = $table_name
            """,
            owner=rdf_owner,
            table_name=rdf_table,
        )
        print(f'\nRDF graph table metadata for {rdf_table_name}:')
        pprint(rdf_table_rows)

        after_rows, _, _ = await driver.execute_query(
            f'SELECT COUNT(*) AS triple_count FROM {rdf_table_name}'
        )
        print(f'\nTriple count after insert in {rdf_table_name}:')
        pprint(after_rows)
    finally:
        await driver.close()

In [ ]:
await create_oracle_graph()

In [ ]:
async def create_oracle_pg_graph() -> None:
    """Create and query a small graph through OraclePGDriver tables."""
    import importlib.util
    from datetime import datetime, timezone
    from uuid import uuid4

    from graphiti_core.driver.oracle_pg_driver import OraclePGDriver
    from graphiti_core.search.search_filters import SearchFilters

    if importlib.util.find_spec('oracledb') is None:
        raise RuntimeError(
            'Missing dependency: oracledb. Install with `uv add oracledb` (or `pip install oracledb`).'
        )

    if not (os.getenv('ORACLE_DSN') or os.getenv('ORACLE_URI')):
        raise RuntimeError('Set ORACLE_DSN or ORACLE_URI before running Oracle PG notebook test.')

    if not os.getenv('ORACLE_USER') or not os.getenv('ORACLE_PASSWORD'):
        raise RuntimeError('Set ORACLE_USER and ORACLE_PASSWORD before running Oracle PG notebook test.')

    driver = OraclePGDriver(log_queries=True, graph_id='NOTEBOOK_DEMO')

    created_at = datetime.now(timezone.utc)
    group_id = f'notebook-pg-demo-{uuid4().hex[:8]}'

    alice = EntityNode(
        uuid=f'entity-{uuid4().hex[:8]}',
        name='Alice',
        group_id=group_id,
        labels=['Person'],
        created_at=created_at,
        summary='Oracle PG demo person node created from the notebook.',
        attributes={'role': 'engineer', 'team': 'graph'},
    )
    bob = EntityNode(
        uuid=f'entity-{uuid4().hex[:8]}',
        name='Bob',
        group_id=group_id,
        labels=['Person'],
        created_at=created_at,
        summary='Second Oracle PG demo person node created from the notebook.',
        attributes={'role': 'manager', 'team': 'graph'},
    )
    works_with = EntityEdge(
        uuid=f'edge-{uuid4().hex[:8]}',
        group_id=group_id,
        source_node_uuid=alice.uuid,
        target_node_uuid=bob.uuid,
        created_at=created_at,
        name='works_with',
        fact='Alice works with Bob on the Graphiti Oracle PG integration.',
        episodes=[],
        valid_at=created_at,
        attributes={'confidence': 0.99, 'source': 'test.ipynb'},
    )

    try:
        connection_rows, _, _ = await driver.execute_query('SELECT 1 AS value FROM dual')
        print('Connection check:')
        pprint(connection_rows)

        await driver.build_indices_and_constraints(delete_existing=True)
        print('\nRan build_indices_and_constraints() for Oracle PG bootstrap.')

        entity_table = driver.table_name('entity_nodes')
        edge_table = driver.table_name('entity_edges')

        before_rows, _, _ = await driver.execute_query(
            f'SELECT COUNT(*) AS node_count FROM {entity_table} WHERE group_id = $group_id',
            group_id=group_id,
        )
        print(f'\nNode count before insert in {entity_table} for group {group_id}:')
        pprint(before_rows)

        await driver.entity_node_ops.save(driver, alice)
        await driver.entity_node_ops.save(driver, bob)
        await driver.entity_edge_ops.save(driver, works_with)

        fetched_alice = await driver.entity_node_ops.get_by_uuid(driver, alice.uuid)
        fetched_edge = await driver.entity_edge_ops.get_by_uuid(driver, works_with.uuid)

        print(f'\nCreated Oracle PG demo graph group: {group_id}')
        pprint(
            {
                'node_uuids': [alice.uuid, bob.uuid],
                'edge_uuid': works_with.uuid,
                'relation': works_with.fact,
                'fetched_alice': fetched_alice.name,
                'fetched_edge': fetched_edge.name,
            }
        )

        node_search = await driver.search_ops.node_fulltext_search(
            driver,
            query='Alice',
            search_filter=SearchFilters(),
            group_ids=[group_id],
            limit=5,
        )
        print('\nFulltext search results for "Alice":')
        pprint([{'uuid': node.uuid, 'name': node.name} for node in node_search])

        after_rows, _, _ = await driver.execute_query(
            (
                f'SELECT '
                f'(SELECT COUNT(*) FROM {entity_table} WHERE group_id = $group_id) AS node_count, '
                f'(SELECT COUNT(*) FROM {edge_table} WHERE group_id = $group_id) AS edge_count '
                'FROM dual'
            ),
            group_id=group_id,
        )
        print(f'\nCounts after insert for group {group_id}:')
        pprint(after_rows)
    finally:
        await driver.close()

In [ ]:
await create_oracle_pg_graph()